# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassaanSaqib/FlyRankAI-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import glob
import subprocess
import pandas as pd
import numpy as np

# Clone my GitHub repo into Colab if it is not already available
repo_dir = "/content/flyrank_repo"

if not os.path.exists(repo_dir):
  subprocess.run(
      [
          "git",
          "clone",
          "-q",
          "https://github.com/HassaanSaqib/FlyRankAI-Internship.git",
          repo_dir,
        ],
      check=True,
      )

# Find the starter dataset
csv_files = glob.glob(f"{repo_dir}/**/*.csv", recursive=True)

df = None
data_path = None

for path in csv_files:
  try:
    temp_df = pd.read_csv(path)
    # Look for the starter search dataset
    if {"impressions_90d", "trend_direction"}.issubset(temp_df.columns):
      df = temp_df
      data_path = path
      break
  except Exception:
    continue

if df is None:
  raise FileNotFoundError("Could not find the starter FlyRank dataset.")

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nAvailable columns:")
print(list(df.columns))

print("\nML task framing:")
print("Lane: Refresh / Content Opportunity Scoring")
print("Primary task type: Scoring and ranking")
print("Unit to rank: Individual content pages")

Dataset loaded successfully.
Rows: 200
Columns: 28

Available columns:
['final_rank', 'content_id', 'client_id', 'final_refresh_score', 'best_model_name', 'best_model_probability', 'baseline_refresh_score', 'confidence', 'suggested_action', 'final_reason_codes', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

ML task framing:
Lane: Refresh / Content Opportunity Scoring
Primary task type: Scoring and ranking
Unit to rank: Individual content pages


My chosen lane is Refresh / Content Opportunity Scoring. I would frame this primarily as a scoring and ranking task. The goal is to assign content pages an opportunity score based on multiple search performance signals and rank them by which pages may deserve human review first. The output would support a content team in prioritizing its work rather than automatically deciding which pages must be refreshed.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["needs_review_proxy"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Target/proxy preview:")
display(
    df[["trend_direction", "needs_review_proxy"]].head(10)
)

print("\nProxy distribution:")
print(df["needs_review_proxy"].value_counts())

print(
    "\n1 = page is marked as downward-trending and may deserve review"
)
print(
    "0 = page is not marked as downward-trending"
)

Target/proxy preview:


,trend_direction,needs_review_proxy
0,down,1
1,down,1
2,down,1
3,down,1
4,down,1
5,down,1
6,down,1
7,down,1
8,down,1
9,down,1



Proxy distribution:
needs_review_proxy
1    197
0      3
Name: count, dtype: int64

1 = page is marked as downward-trending and may deserve review
0 = page is not marked as downward-trending


My provisional target is whether a page may need review, using its observed trend direction as a simple proxy in the starter data. For this initial framing, a downward-trending page is assigned a needs_review_proxy value of 1, while a page that is not downward-trending is assigned 0.

In the starter dataset, 197 out of 200 pages match this downward-trend proxy, while only 3 pages do not. This shows that the proxy is highly imbalanced and may not be sufficient as a final target by itself. I would therefore treat it as an initial proxy for exploring the task, not as proof that a page needs to be refreshed or that refreshing it would improve performance

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Success metric: Precision@20

k = min(20, len(df))

# Simple baseline score: rank pages by impressions

results = df[["impressions_90d", "needs_review_proxy"]].copy()

top_k = results.sort_values(

    by="impressions_90d",

    ascending=False

).head(k)

precision_at_20 = top_k["needs_review_proxy"].mean()

print(f"Precision@{k}: {precision_at_20:.3f}")

print(

    f"{precision_at_20 * 100:.1f}% of the top {k} pages "

    "ranked by impressions match the current review proxy."

)

Precision@20: 1.000
100.0% of the top 20 pages ranked by impressions match the current review proxy.


I would use Precision@K as an initial success metric because the goal is to produce a useful ranked list for a content team with limited time. Precision@K measures how many of the top K recommended pages match the defined review target or proxy.

As an illustrative baseline, I ranked pages by 90-day impressions and measured Precision@20. The result was 1.000, meaning that all 20 of the highest-impression pages matched the current downward-trend review proxy. However, because 197 of the 200 pages match this proxy, this result should not be interpreted as evidence of a strong predictive model. It mainly shows that a more selective target or proxy will be needed to meaningfully evaluate a future ranking system

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Select useful columns that actually exist in the dataset
candidate_columns = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "trend_direction",
    "needs_review_proxy"
]

available_columns = [
    col for col in candidate_columns
    if col in df.columns
]

lane_slice = df[available_columns].copy()

print("Unit of analysis: ONE ROW = ONE CONTENT PAGE")
print("Number of page rows:", len(lane_slice))
print("Columns shown:", available_columns)

display(lane_slice.head(10))

Unit of analysis: ONE ROW = ONE CONTENT PAGE
Number of page rows: 200
Columns shown: ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'content_age_days', 'trend_direction', 'needs_review_proxy']


,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,trend_direction,needs_review_proxy
0,12834,6,0.05,6.8,165,down,1
1,8064,6,0.07,3.8,139,down,1
2,2498,0,0.00,10.1,165,down,1
3,13790,16,0.12,8.2,139,down,1
4,3393,3,0.09,3.6,131,down,1
5,5811,4,0.07,6.4,144,down,1
6,1622,2,0.12,3.1,139,down,1
7,2621,0,0.00,12.8,144,down,1
8,1597,2,0.13,2.7,131,down,1
9,3867,2,0.05,27.5,131,down,1


The unit of analysis is an individual content page, meaning that one row in the dataframe represents one content page. The starter dataset contains 200 page rows.

For this initial lane slice, I examined available page-level signals including 90-day impressions, 90-day clicks, CTR, average position, content age, and trend direction, together with the provisional needs_review_proxy. These features show that pages can have different combinations of visibility, engagement, ranking position, age, and performance trends. The eventual goal is to use relevant signals to score and rank pages for human review

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
comparison_columns = [
col for col in [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "trend_direction"
    ]
    if col in df.columns
]

comparison = df[comparison_columns].copy()

# Simple fixed rule:
# review every page with impressions above the median
median_impressions = df["impressions_90d"].median()

comparison["fixed_rule_review"] = (
    df["impressions_90d"] >= median_impressions
)

print(
    f"Fixed rule example: review pages with impressions >= "
    f"{median_impressions:.0f}"
)

print(
    "Pages selected by fixed rule:",
    comparison["fixed_rule_review"].sum()
)

print("\nSample of pages selected by the same fixed rule:")
display(
    comparison[
        comparison["fixed_rule_review"]
        ].head(10)
)

print(
    "\nThese pages may have similar visibility but different CTR, "
    "positions, clicks, and trend directions. A scoring/ranking "
    "approach can consider several signals together instead of "
    "depending on one threshold."
)

Fixed rule example: review pages with impressions >= 4783
Pages selected by fixed rule: 100

Sample of pages selected by the same fixed rule:


,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,fixed_rule_review
0,12834,6,0.05,6.8,down,True
1,8064,6,0.07,3.8,down,True
3,13790,16,0.12,8.2,down,True
5,5811,4,0.07,6.4,down,True
14,5541,8,0.14,11.2,down,True
16,9239,3,0.03,5.6,down,True
20,27437,19,0.07,7.2,down,True
22,10575,14,0.13,1.9,down,True
24,15616,9,0.06,8.8,stable,True
27,11926,5,0.04,4.4,down,True



These pages may have similar visibility but different CTR, positions, clicks, and trend directions. A scoring/ranking approach can consider several signals together instead of depending on one threshold.


A single fixed rule may be too limited because content opportunity depends on several signals that can interact with each other. For example, two pages with high impressions may have very different click-through rates, average positions, click counts, and trend directions.

As a simple comparison, I tested a fixed rule that selects pages with 90-day impressions greater than or equal to the dataset median of 4,783 impressions. This rule selected 100 of the 200 pages, but it treats all selected pages similarly even though their other performance signals differ. A scoring or ranking approach can consider multiple signals together and potentially produce a more useful order for review. The purpose would still be decision support: the final ranking would help a human decide where to investigate first rather than automatically determining that a page must be refreshed.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.